## Complete Reuters Test Set

* Tasks
    * ~~Combine the two files of scraped data~~
    * ~~Fill missing data~~
        * ~~Content not available = drop~~
        * ~~Decide on what to do if the title or description is missing~~
        * ~~For date, take the next date (eliminates risk of peeking into the future)~~
    * ~~Check the discrepancy for the overlapping data, then combine~~ <- My scraped data is around 3 times denser
        * ~~From that Kaggle data there is a timespan from 2018 up to July 2020
    * ~~Format date properly~~
    * Visual dipiction, articles-per-day, maybe topic mapping
    * Save for later analysis

In [1]:
import pandas as pd

In [2]:
first = pd.read_csv('../data/reuters/2021_04_11_2020_11_21.csv')
second = pd.read_csv('../data/reuters/prior_to_2020_11_21.csv')

reuters_articles = pd.concat([first, second], ignore_index=True)

In [3]:
reuters_articles.tail(5)

,Unnamed: 0,title,description,date
44001,31014,no_title,"Our apologies, the content you requested canno...",no_date
44002,31015,no_title,"Our apologies, the content you requested canno...",no_date
44003,31016,no_title,"Our apologies, the content you requested canno...",no_date
44004,31017,no_title,"Our apologies, the content you requested canno...",no_date
44005,31018,no_title,"Our apologies, the content you requested canno...",no_date


In [4]:
while 'no_title' == reuters_articles.iloc[-1]['title']:
    reuters_articles = reuters_articles.drop(reuters_articles.index[-1])
    
reuters_articles.iloc[-1]

Unnamed: 0                                                 29314
title          White House to release Obama's 2016 budget on ...
description                                       no_description
date                                                     no_date
Name: 42301, dtype: object

In [5]:
reuters_articles = reuters_articles.drop(['Unnamed: 0'], axis=1)

In [6]:
reuters_articles.tail(5)

,title,description,date
42297,Egypt deep cleans pyramids site emptied of tou...,Egypt began deep cleaning the area around the ...,Mar 25 2020
42298,Yemen's Houthi leader says Saudi-led coalition...,A leader of the Iran-aligned Yemen's Houthi mo...,Mar 25 2020
42299,"Obama touts auto bailout success, Michigan wor...","President Barack Obama, on a trip to unveil el...",no_date
42300,U.S. House fails to pass Republican bill dilut...,no_description,no_date
42301,White House to release Obama's 2016 budget on ...,no_description,no_date


In [7]:
# I'm sorry for the python sins that I am commiting I just couldn't see how I would be able to do this with zip, apply.
# For this usecase I'm not aware of a 'cythonized' method
for i in range(reuters_articles.index[-1] + 1):
    row = reuters_articles.iloc[i]
    # Take the date of the previous row -> won't accidentally date something earlier (There was news every day)
    if 'no_date' == row['date']:
        reuters_articles.iloc[i]['date'] = reuters_articles.iloc[i - 1]['date']
        
    if 'no_description' == row['description']:
        reuters_articles.iloc[i]['description'] = row['title']
        
reuters_articles.tail(3)

,title,description,date
42299,"Obama touts auto bailout success, Michigan wor...","President Barack Obama, on a trip to unveil el...",Mar 25 2020
42300,U.S. House fails to pass Republican bill dilut...,U.S. House fails to pass Republican bill dilut...,Mar 25 2020
42301,White House to release Obama's 2016 budget on ...,White House to release Obama's 2016 budget on ...,Mar 25 2020


In [8]:
# Shouldn't be the case
reuters_articles = reuters_articles.drop_duplicates(ignore_index=True)

In [9]:
reuters_articles.tail(3)

,title,description,date
42222,"Obama touts auto bailout success, Michigan wor...","President Barack Obama, on a trip to unveil el...",Mar 25 2020
42223,U.S. House fails to pass Republican bill dilut...,U.S. House fails to pass Republican bill dilut...,Mar 25 2020
42224,White House to release Obama's 2016 budget on ...,White House to release Obama's 2016 budget on ...,Mar 25 2020


In [10]:
# Headlines, Time, Description
third_party_articles = pd.read_csv('../data/kaggle/financial-news-headlines/reuters_headlines.csv')
len(third_party_articles)

32770

In [11]:
third_party_articles.head(3)

,Headlines,Time,Description
0,TikTok considers London and other locations fo...,Jul 18 2020,TikTok has been in discussions with the UK gov...
1,Disney cuts ad spending on Facebook amid growi...,Jul 18 2020,Walt Disney has become the latest company to ...
2,Trail of missing Wirecard executive leads to B...,Jul 18 2020,Former Wirecard chief operating officer Jan M...


In [12]:
third_party_articles.columns = ['title', 'date', 'description']
third_party_articles.head(3)

,title,date,description
0,TikTok considers London and other locations fo...,Jul 18 2020,TikTok has been in discussions with the UK gov...
1,Disney cuts ad spending on Facebook amid growi...,Jul 18 2020,Walt Disney has become the latest company to ...
2,Trail of missing Wirecard executive leads to B...,Jul 18 2020,Former Wirecard chief operating officer Jan M...


In [13]:
third_party_articles = third_party_articles[['title', 'description', 'date']]
third_party_articles

,title,description,date
0,TikTok considers London and other locations fo...,TikTok has been in discussions with the UK gov...,Jul 18 2020
1,Disney cuts ad spending on Facebook amid growi...,Walt Disney has become the latest company to ...,Jul 18 2020
2,Trail of missing Wirecard executive leads to B...,Former Wirecard chief operating officer Jan M...,Jul 18 2020
3,Twitter says attackers downloaded data from up...,Twitter Inc said on Saturday that hackers were...,Jul 18 2020
4,U.S. Republicans seek liability protections as...,A battle in the U.S. Congress over a new coron...,Jul 17 2020
...,...,...,...
32765,Malaysia says never hired British data firm at...,The Malaysian government and the ruling party ...,Mar 20 2018
32766,Prosecutors search Volkswagen headquarters in ...,German prosecutors said on Tuesday they had se...,Mar 20 2018
32767,McDonald's sets greenhouse gas reduction targets,McDonald's Corp on Tuesday announced an approv...,Mar 20 2018
32768,Pratt & Whitney to deliver spare A320neo engin...,Pratt & Whitney will soon begin deliveries of ...,Mar 20 2018


In [14]:
third_party_articles['date'] = pd.to_datetime(third_party_articles['date'])
reuters_articles['date']  = pd.to_datetime(reuters_articles['date'])

In [15]:
reuters_june = reuters_articles.loc[(reuters_articles['date'].dt.year==2020) & (reuters_articles['date'].dt.month==6)]
reuters_june

,title,description,date
29911,China says suspects arrested by HK security of...,Suspects arrested by the mainland's new office...,2020-06-30
29912,Germany's confirmed coronavirus cases rise by ...,The number of confirmed coronavirus cases in G...,2020-06-30
29913,Australia to sharply increase defence spending...,Australia will boost defence spending by 40% o...,2020-06-30
29914,Highlights: China unveils details of national ...,New Hong Kong security laws came into effect o...,2020-06-30
29915,Australia's second largest city orders 36 subu...,Authorities on Tuesday ordered the lockdown of...,2020-06-30
...,...,...,...
33616,UK is following scientific advice on cautious ...,The British government is following scientific...,2020-06-01
33617,Japan parliament to debate second extra budget...,Japan's government will submit to parliament e...,2020-06-01
33618,U.S. senators vote to bolster travel security ...,The U.S. Senate voted on Thursday to bolster t...,2020-06-01
33619,Trump cancels California event to stay in New ...,Trump cancels California event to stay in New ...,2020-06-01


In [16]:
third_party_june = third_party_articles.loc[(third_party_articles['date'].dt.year==2020) & (third_party_articles['date'].dt.month==6)]
third_party_june

,title,description,date
893,Google postpones U.S. office reopening to Sept...,Alphabet Inc's Google said late on Tuesday it...,2020-06-30
894,Carlyle to buy 25% of Bharti Airtel's data cen...,Carlyle will buy a 25% stake in Indian telecom...,2020-06-30
895,SocGen's Australian securities arm pleads guil...,Australia's corporate regulator on Wednesday s...,2020-06-30
896,"China's factory activity expands, but job loss...",China's factory activity grew at a faster clip...,2020-06-30
897,California accuses Cisco of job discrimination...,California regulators sued Cisco Systems Inc ...,2020-06-30
...,...,...,...
2200,"Worst may be over for euro zone factories, rec...",Euro zone manufacturers appear to have passed ...,2020-06-01
2201,Hyundai Motor's May sales fall sharply year-on...,South Korea's Hyundai Motor Co said on Monday...,2020-06-01
2202,Analysts' View: Impact of the U.S. protests on...,National Guard troops have been deployed in 15...,2020-06-01
2203,Workers nervously eye return to Lear's coronav...,Lear Corp is implementing costly safety measur...,2020-06-01


### Evaluation of Data
The data from Kaggle has a longer timeframe with mine being denser. Combining the sets is probably not the worst, the training data (up to the end of 2020) will be weighted more highly with more recent articles.

In [17]:
reuters_data = pd.concat([reuters_articles, third_party_articles], ignore_index=True)
reuters_data

,title,description,date
0,Amid COVID-19 concerns and multiple candidates...,LIMA (Reuters) -Peru's presidential candidates...,2021-04-11
1,Ecuador weighs returning to socialism in presi...,QUITO (Reuters) -Ecuadoreans voted in a presid...,2021-04-11
2,Saudi-led coalition intercepts explosive-laden...,The Saudi-led coalition fighting in Yemen on S...,2021-04-11
3,Vote counting in Benin after election marked b...,COTONOU (Reuters) -Vote counting began in Beni...,2021-04-11
4,"Top U.S. diplomat criticizes China, says 'need...",China's failure to provide access to global he...,2021-04-11
...,...,...,...
74990,Malaysia says never hired British data firm at...,The Malaysian government and the ruling party ...,2018-03-20
74991,Prosecutors search Volkswagen headquarters in ...,German prosecutors said on Tuesday they had se...,2018-03-20
74992,McDonald's sets greenhouse gas reduction targets,McDonald's Corp on Tuesday announced an approv...,2018-03-20
74993,Pratt & Whitney to deliver spare A320neo engin...,Pratt & Whitney will soon begin deliveries of ...,2018-03-20


In [18]:
reuters_data.drop_duplicates(inplace=True, ignore_index=True)
reuters_data

,title,description,date
0,Amid COVID-19 concerns and multiple candidates...,LIMA (Reuters) -Peru's presidential candidates...,2021-04-11
1,Ecuador weighs returning to socialism in presi...,QUITO (Reuters) -Ecuadoreans voted in a presid...,2021-04-11
2,Saudi-led coalition intercepts explosive-laden...,The Saudi-led coalition fighting in Yemen on S...,2021-04-11
3,Vote counting in Benin after election marked b...,COTONOU (Reuters) -Vote counting began in Beni...,2021-04-11
4,"Top U.S. diplomat criticizes China, says 'need...",China's failure to provide access to global he...,2021-04-11
...,...,...,...
74871,Malaysia says never hired British data firm at...,The Malaysian government and the ruling party ...,2018-03-20
74872,Prosecutors search Volkswagen headquarters in ...,German prosecutors said on Tuesday they had se...,2018-03-20
74873,McDonald's sets greenhouse gas reduction targets,McDonald's Corp on Tuesday announced an approv...,2018-03-20
74874,Pratt & Whitney to deliver spare A320neo engin...,Pratt & Whitney will soon begin deliveries of ...,2018-03-20


Not as many duplicates as I was expecting. Will need to reorder by date again.

In [29]:
reuters_data.tail(60) # there were a lot of articles on the 20th

,title,description,date
74816,Facebook took years to clamp down on developer...,A former Facebook operations manager told a Br...,2018-03-21
74817,"Deere & Co fears hit from Trump tariffs, retal...",U.S. tractor maker Deere & Co is bracing for ...,2018-03-21
74818,Stocks end modestly lower after Fed hikes rate...,"U.S. stocks ended slightly lower on Wednesday,...",2018-03-21
74819,UBS in $230 million settlement of New York mor...,UBS AG has reached a $230 million settlement ...,2018-03-21
74820,IMF's Lagarde urges G20 to avoid 'exceptional'...,International Monetary Fund Managing Director ...,2018-03-20
74821,Exxon eyes Gulf of Mexico plastics plant to me...,Exxon Mobil Corp said on Tuesday it was plann...,2018-03-20
74822,G20 financial leaders say need more dialogue o...,The world's financial leaders reaffirmed on Tu...,2018-03-20
74823,Index provider MSCI says it is reviewing Faceb...,MSCI Inc on Tuesday said it is looking into th...,2018-03-20
74824,UK's Cambridge University questions Facebook a...,Cambridge University said it wanted Facebook t...,2018-03-20
74825,Exclusive: Salesforce in advanced talks to buy...,Salesforce.com Inc is in advanced discussions...,2018-03-20


In [ ]:
reuters_data.